# 04. LoRA 기반 사전학습 언어모델 파인튜닝

- 모델: `klue/bert-base`
- 방법: LoRA(Low-Rank Adaptation)
- 입력 데이터: `data/processed/train_processed.csv`, `data/processed/valid_processed.csv`
- 분류 기준: 정상 댓글 `0`, 악성 댓글 `1`
- 저장 결과: metrics, predictions, confusion matrix

## LoRA 사용 이유

LoRA는 전체 BERT 파라미터를 모두 학습하지 않고, attention layer 일부에 작은 adapter 파라미터를 추가해 학습하는 효율적인 파인튜닝 기법이다. 일반 full fine-tuning보다 학습해야 하는 파라미터 수가 적어 자원 효율성이 높고, 성능 비교 실험에도 활용할 수 있다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
MODEL_NAME = "klue/bert-base"
MAX_LENGTH = 128
OUTPUT_DIR = Path("../model/klue_bert_lora_finetuned")
REPORT_DIR = Path("../reports")

set_seed(42)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 데이터 불러오기

In [ ]:
train_df = pd.read_csv("../data/processed/train_processed.csv")
valid_df = pd.read_csv("../data/processed/valid_processed.csv")

display(train_df.head())
print(train_df.shape, valid_df.shape)
print(train_df["label"].value_counts())

## Dataset 클래스 정의

In [ ]:
class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

## 평가 지표 함수

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }


def save_confusion_matrix(y_true, y_pred, output_path):
    plt.rcParams["font.family"] = "Malgun Gothic"
    plt.rcParams["axes.unicode_minus"] = False

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["정상", "악성"],
        yticklabels=["정상", "악성"],
    )
    plt.title("KLUE BERT LoRA Confusion Matrix")
    plt.xlabel("예측")
    plt.ylabel("실제")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()

## 토크나이저와 LoRA 모델 준비

In [ ]:
train_texts = train_df["comments"].astype(str).tolist()
valid_texts = valid_df["comments"].astype(str).tolist()
train_labels = train_df["label"].astype(int).tolist()
valid_labels = valid_df["label"].astype(int).tolist()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer)
valid_dataset = HateSpeechDataset(valid_texts, valid_labels, tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## LoRA 파인튜닝

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="../reports/logs_lora",
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 검증 데이터 예측 및 결과 저장

In [ ]:
predictions = trainer.predict(valid_dataset)
y_pred = predictions.predictions.argmax(axis=-1)
y_true = predictions.label_ids

metrics = {
    "model": MODEL_NAME,
    "method": "LoRA",
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
}

pd.DataFrame([metrics]).to_csv(REPORT_DIR / "KLUE_BERT_LoRA_metrics.csv", index=False, encoding="utf-8-sig")

result_df = valid_df.copy()
result_df["predicted"] = y_pred
result_df["correct"] = result_df["predicted"] == result_df["label"]
result_df.to_csv(REPORT_DIR / "KLUE_BERT_LoRA_predictions.csv", index=False, encoding="utf-8-sig")

save_confusion_matrix(y_true, y_pred, REPORT_DIR / "KLUE_BERT_LoRA_confusion_matrix.png")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(metrics)
print(classification_report(y_true, y_pred, target_names=["정상", "악성"]))